# MercadoLab — validação executável

Este notebook demonstra o uso do MercadoLab em três níveis:

1. validação da API pública e execução de um cenário mínimo;
2. execução do cenário anual de referência com coleta de métricas;
3. verificação de reprodutibilidade e geração de gráficos.

O MercadoLab fornece uma implementação padrão baseada em livro de ofertas com prioridade preço-tempo. Estratégias dos participantes e políticas de geração de ordens permanecem externas à framework.

> **Escopo da evidência:** os resultados deste notebook verificam integração, consistência operacional e reprodutibilidade. Eles não constituem, isoladamente, validação econômica, calibração com dados reais ou comprovação de fatos estilizados.

## 1. Preparação do ambiente

No Google Colab, o notebook clona o repositório e fixa o commit de referência. Em execução local, ele procura a raiz do repositório entre o diretório atual e seus diretórios-pai.

As dependências são instaladas diretamente do repositório; nenhuma publicação no PyPI é necessária.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

REPOSITORIO = "https://github.com/Hargenx/mercadolab.git"
COMMIT_REFERENCIA = "6ab9db282acd9268ea1877bae04ab2d7f09eeee0"

def executar(comando: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    ambiente = os.environ.copy()
    ambiente["PYTHONUTF8"] = "1"
    return subprocess.run(
        comando,
        cwd=cwd,
        env=ambiente,
        check=True,
        text=True,
        encoding="utf-8",
        capture_output=True,
    )


if "google.colab" in sys.modules:
    raiz_repositorio = Path("/content/mercadolab")
    if not raiz_repositorio.exists():
        executar(["git", "clone", REPOSITORIO, str(raiz_repositorio)])
    executar(["git", "fetch", "origin"], cwd=raiz_repositorio)
    executar(["git", "checkout", COMMIT_REFERENCIA], cwd=raiz_repositorio)
else:
    candidatos = (Path.cwd(), *Path.cwd().parents)
    raiz_repositorio = next(
        (pasta for pasta in candidatos if (pasta / "pyproject.toml").exists()),
        None,
    )
    if raiz_repositorio is None:
        raise RuntimeError("Execute o notebook dentro do repositório MercadoLab.")

executar(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{raiz_repositorio}[dev,data,plot]",
    ]
)
os.chdir(raiz_repositorio)

caminho_src = str((raiz_repositorio / "src").resolve())

if caminho_src in sys.path:
    sys.path.remove(caminho_src)

sys.path.insert(0, caminho_src)

commit_executado = executar(
    ["git", "rev-parse", "HEAD"],
    cwd=raiz_repositorio,
).stdout.strip()

print("Repositório:", raiz_repositorio)
print("Commit executado:", commit_executado)

**Registro do ambiente.**

In [ ]:
import platform

import matplotlib
import pandas

print("Ambiente:", "Google Colab" if "google.colab" in sys.modules else "Local")
print("Python:", sys.version)
print("Plataforma:", platform.platform())
print("Pandas:", pandas.__version__)
print("Matplotlib:", matplotlib.__version__)

## 2. API pública e cenário mínimo

A superfície pública concentra os componentes usados para construir cenários. O exemplo mínimo cria um ativo, dois participantes, duas ordens compatíveis e executa um tick.

In [ ]:
import importlib

sys.modules.pop("mercadolab", None)
mercadolab = importlib.import_module("mercadolab")

arquivo_carregado = Path(mercadolab.__file__).resolve()
arquivo_esperado = (
    raiz_repositorio / "src" / "mercadolab" / "__init__.py"
).resolve()

print("MercadoLab carregado de:", arquivo_carregado)
assert arquivo_carregado == arquivo_esperado

SIMBOLOS_ESPERADOS = {
    "Ativo",
    "Carteira",
    "Investidor",
    "LadoOrdem",
    "LivroDeOfertas",
    "Mercado",
    "Ordem",
    "Posicao",
    "Simulacao",
    "StatusOrdem",
    "Tempo",
    "TipoAtivo",
    "TipoOrdem",
    "Transacao",
}

assert set(mercadolab.__all__) == SIMBOLOS_ESPERADOS
print("API pública validada:", ", ".join(sorted(SIMBOLOS_ESPERADOS)))

In [ ]:
resultado_testes = executar(
    [sys.executable, "-m", "pytest"],
    cwd=raiz_repositorio,
)
print(resultado_testes.stdout)

In [ ]:
resultado_minimo = executar(
    [sys.executable, "-m", "mercadolab.scenarios.exemplo_minimo_fii"],
    cwd=raiz_repositorio,
)
print(resultado_minimo.stdout)

## 3. Cenário anual e coleta de métricas

O cenário de referência utiliza 100 investidores, 252 ticks e seed 42. A política de geração de ordens pertence ao cenário, não ao núcleo da framework.

As métricas registradas por tick são:

- quantidade de ordens;
- quantidade de transações;
- volume financeiro;
- preço médio das transações;
- último preço negociado.

In [ ]:
resultado_metricas = executar(
    [sys.executable, "-m", "mercadolab.scenarios.exemplo_coleta_metricas"],
    cwd=raiz_repositorio,
)
print(resultado_metricas.stdout)

In [ ]:
import pandas as pd

arquivo_csv = raiz_repositorio / "metricas_simulacao_anual.csv"
metricas = pd.read_csv(arquivo_csv)

resumo = {
    "linhas": len(metricas),
    "ticks_unicos": int(metricas["tick"].nunique()),
    "primeiro_tick": int(metricas["tick"].min()),
    "ultimo_tick": int(metricas["tick"].max()),
    "total_transacoes": int(metricas["transacoes"].sum()),
    "volume_total": float(metricas["volume"].sum()),
    "media_ordens": round(float(metricas["ordens"].mean()), 2),
    "media_transacoes": round(float(metricas["transacoes"].mean()), 2),
}

assert resumo["linhas"] == 252
assert resumo["ticks_unicos"] == 252
assert resumo["primeiro_tick"] == 0
assert resumo["ultimo_tick"] == 251
assert resumo["total_transacoes"] == 11243
assert resumo["volume_total"] == 2031928.0

pd.Series(resumo, name="valor").to_frame()

## 4. Verificação de reprodutibilidade

A coleta é executada novamente com a mesma versão, configuração e seed. A igualdade do SHA-256 do CSV demonstra que as duas execuções produziram o mesmo artefato tabular.

In [ ]:
def sha256(caminho: Path) -> str:
    return hashlib.sha256(caminho.read_bytes()).hexdigest()


hash_primeira_execucao = sha256(arquivo_csv)

segunda_execucao = executar(
    [sys.executable, "-m", "mercadolab.scenarios.exemplo_coleta_metricas"],
    cwd=raiz_repositorio,
)
hash_segunda_execucao = sha256(arquivo_csv)

assert hash_primeira_execucao == hash_segunda_execucao

print("SHA-256 da primeira execução:", hash_primeira_execucao)
print("SHA-256 da segunda execução:", hash_segunda_execucao)
print("CSV reproduzido de forma idêntica.")

## 5. Geração e inspeção dos gráficos

Os gráficos são derivados do CSV validado. Eles servem como evidência visual do funcionamento do cenário e não devem ser interpretados como comprovação automática de realismo econômico.

In [ ]:
resultado_graficos = executar(
    [sys.executable, "-m", "mercadolab.scenarios.exemplo_graficos_metricas"],
    cwd=raiz_repositorio,
)
print(resultado_graficos.stdout)

from IPython.display import Image, display

arquivos_graficos = [
    "ordens_por_tick.png",
    "transacoes_por_tick.png",
    "volume_por_tick.png",
    "preco_medio_por_tick.png",
    "ultimo_preco_por_tick.png",
]

diretorio_graficos = raiz_repositorio / "graficos_metricas"
for nome_arquivo in arquivos_graficos:
    caminho = diretorio_graficos / nome_arquivo
    assert caminho.exists()
    print(nome_arquivo)
    display(Image(filename=str(caminho)))

## 6. Interpretação e limites

Este notebook demonstra que:

- a API pública pode ser importada e usada por um cenário externo;
- ordens, pareamento, transações, carteiras e tempo funcionam de forma integrada;
- o cenário anual completa 252 ticks;
- as métricas podem ser exportadas e visualizadas;
- a mesma versão, configuração e seed reproduzem o mesmo CSV.

Ele não demonstra, por si só:

- calibração com um mercado real;
- emergência de fatos estilizados;
- superioridade de uma estratégia;
- validade econômica geral do cenário.

Essas questões exigem desenhos experimentais próprios, construídos sobre a framework e apresentados separadamente na dissertação.

In [ ]:
print("Validação executável concluída.")
print("Commit:", commit_executado)
print("Ticks:", resumo["linhas"])
print("Transações:", resumo["total_transacoes"])
print("Volume:", f'{resumo["volume_total"]:.2f}')
print("SHA-256 do CSV:", hash_segunda_execucao)